<a href="https://colab.research.google.com/github/ereznaamansapienza/mnlp/blob/ananas/mnlp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

## Config

In [29]:
import torch
import gc
import os
import json
import numpy as np
import pandas as pd
import random
from datasets import load_dataset, Dataset, load_from_disk

from transformers import AutoModelForCausalLM, AutoTokenizer

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from typing import Dict, List, Tuple, Callable, Optional
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

from tqdm.auto import tqdm

In [30]:
torch.cuda.empty_cache()
gc.collect()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

ds_url = "sapienzanlp-course-materials/hw-mnlp-2026"

cuda


In [31]:
SEED = 42

def set_seed(seed: int = 42):
  random.seed(seed)
  np.random.seed(seed)

  torch.manual_seed(seed)

  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

set_seed(SEED)

## DataModule

In [4]:
class DataModule:
    def __init__(self, dev_size: float = 0.2, seed: int = 42):
        self.ds = None
        self.dev_size = dev_size
        self.seed = seed

    # Loading
    def load(self, url: str):
        self.ds = load_dataset(url)
        return self.ds

    def to_dataframe(self, split_name: str) -> pd.DataFrame:
        return pd.DataFrame(self.ds[split_name])

    # Splits
    def build_train_val_split(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        train_df_full = self.to_dataframe("train")
        train_df, val_df = train_test_split(
            train_df_full,
            test_size=self.dev_size,
            random_state=self.seed,
            shuffle=True,
        )
        return train_df, val_df

    def build_cv_splits(
        self,
        n_folds: int = 5,
        split_name: str = "train",
        stratify_col: str = "wikipedia_title",
    ) -> List[Tuple[pd.DataFrame, pd.DataFrame]]:
        df = self.to_dataframe(split_name).reset_index(drop=True)

        if stratify_col is not None:
            kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=self.seed)
            split_iter = kf.split(df, df[stratify_col])
        else:
            kf = KFold(n_splits=n_folds, shuffle=True, random_state=self.seed)
            split_iter = kf.split(df)

        folds = []
        for train_idx, val_idx in split_iter:
            folds.append((
                df.iloc[train_idx].reset_index(drop=True),
                df.iloc[val_idx].reset_index(drop=True),
            ))
        return folds

    # Dataset builders
    def build_full_train_dataset(
        self,
        dataset_factory: Callable,
        **factory_kwargs,
    ) -> Tuple[pd.DataFrame, Dataset]:
        """Builds a dataset from the entire training split with no val holdout."""
        train_df = self.to_dataframe("train")
        dataset = dataset_factory(train_df, **factory_kwargs)
        return train_df, dataset

    def build_train_val_datasets(
        self,
        dataset_factory: Callable,
        **factory_kwargs,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, Dataset, Dataset]:
        """Splits train into train/val and builds datasets for both."""
        train_df, val_df = self.build_train_val_split()
        train_dataset = dataset_factory(train_df, **factory_kwargs)
        val_dataset = dataset_factory(val_df,   **factory_kwargs)
        return train_df, val_df, train_dataset, val_dataset


## LM Generators

In [38]:
class QALLM():
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model = self.load_model()
        self.tokenizer = self.load_tokenizer()

    def load_model(self):
        return AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype="auto",
            device_map="auto"
        )

    def load_tokenizer(self):
        return AutoTokenizer.from_pretrained(self.model_name)

    def prepare_model_input(self, prompt, thinking):
        messages = [
            {"role": "user", "content": prompt}
        ]
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=thinking # Switches between thinking and non-thinking modes. Default is True
        )
        return self.tokenizer([text], return_tensors="pt").to(self.model.device)

    def generate_answer(self, prompt, thinking):
        model_inputs = self.prepare_model_input(prompt, thinking)

        # conduct text completion
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=64
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

        # parsing thinking content
        try:
            # rindex finding 151668 (</think>)
            index = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            index = 0

        thinking_content = self.tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
        content = self.tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

        return content, thinking_content


In [ ]:
class SmolLM():
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model = self.load_model()
        self.tokenizer = self.load_tokenizer()

    def load_model(self):
        return AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype="auto",
            device_map="auto"
        )

    def load_tokenizer(self):
        return AutoTokenizer.from_pretrained(self.model_name)

    def generate_answer(self, prompt):
      model_inputs = self.tokenizer.encode(prompt, return_tensors="pt").to(device)

      outputs = model.generate(inputs)

      return self.tokenizer.decode(outputs[0])

## RetrievalStore

In [6]:
class RetrievalStore:
    def __init__(self, top_k: int = 3):
      self.top_k = top_k

    def load_rankings_jsonl(self, path: str) -> Dict[str, List[int]]:
        rankings = {}

        with open(path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()

                if not line:
                    continue

                obj = json.loads(line)

                query_id, ranking = next(iter(obj.items()))

                rankings[str(query_id)] = [int(i) for i in ranking]

        return rankings

    def attach_rankings_from_jsonl(self, df: pd.DataFrame, path: str) -> pd.DataFrame:
        df = df.copy()
        rankings = self.load_rankings_jsonl(path)

        # Series.map accepts a dict-like mapping and maps values by key
        df["ranking"] = df["query_id"].astype(str).map(rankings)

        missing = df[df["ranking"].isna()]["query_id"].tolist()

        if missing:
            raise ValueError(
                f"Missing rankings for {len(missing)} query_ids. "
                f"First missing examples: {missing[:5]}"
            )

        df["retrieved_chunks"] = df["ranking"].apply(
            lambda ranking: list(ranking[:self.top_k])
        )

        self._validate_indices(df)

        return df

    def _validate_indices(self, df: pd.DataFrame) -> None:
        for row in df.itertuples():
            n_candidates = len(row.candidate_chunks)

            for idx in row.retrieved_chunks:
                if idx < 0 or idx >= n_candidates:
                    raise ValueError(
                        f"Invalid chunk index {idx} for query_id={row.query_id}. "
                        f"n_candidates={n_candidates}"
                    )

    def get_rag_indices(self, row) -> List[int]:
      return list(row.retrieved_chunks[:self.top_k])

    def get_oracle_indices(self, row) -> List[int]:
      retrieved = list(row.retrieved_chunks[:self.top_k])
      correct_idx = int(row.answer_pos)

      if correct_idx in retrieved:
          retrieved.remove(correct_idx)
      else:
          retrieved = retrieved[:-1]

      return [correct_idx] + retrieved


## PromptBuilder

In [50]:
class PromptBuilder:
  def __init__(self, intro="Given the following information:", command="Reply to this question:"):
    self.intro = intro
    self.command = command

  def build_baseline_prompt(self, query: str) -> str:
    return f"""
      Reply to this question in the shortest possible valid way, do not explain the answer:
      {query}
      """

  def build_zero_shot_prompt(self, query: str, chunks: List[str]) -> str:
    numbered_chunks = "\n".join(
          f"{i + 1}. {chunk}" for i, chunk in enumerate(chunks)
      )

    return self.intro + "\n" + numbered_chunks + "\n" + self.command + "\n" + query

In [9]:
evaluation_prompt = """
You are evaluating a generated answer for a question-answering task.

Question:
{query}

Gold short answer:
{short_answer}

Generated answer:
{generated_answer}

Evaluation criterion:
Return 1 if the generated answer contains the gold short answer in any valid form.
Valid forms include paraphrases, spelling variants, number-word equivalence, or a longer sentence that clearly contains the correct answer.
Return 0 if the generated answer does not contain the correct answer, contradicts it, gives a different entity/date/place/person, or is too vague.

Important:
- Do not reward an answer only because it is fluent.
- Do not use external knowledge.
- Judge only whether the generated answer expresses the gold short answer.
- Output only valid JSON.

Output format:
{"score": 0}
or
{"score": 1}
"""

## AnswerEvaluator

In [ ]:
class AnswerEvaluator:
  @staticmethod
  def get_EM(generated_answer, short_answer):
    return int(generated_answer == short_answer)

  @staticmethod
  def get_subEM(generated_answer, short_answer):
    return int(generated_answer in short_answer)

  # def get_METEOR(generated_answer, short_answer):


## LLMJudge

## AgreementAnalyzer

## ExperimentRunner

In [34]:
class ExperimentRunner:
  def __init__(
        self,
        prompt_builder,
        retrieval_store,
        top_k: int = 3,
        seed: int = 42,
    ):
    self.prompt_builder = prompt_builder
    self.retrieval_store = retrieval_store
    self.top_k = top_k

  def build_prompt_from_row(self, row, setting: str) -> Tuple[str, List[int]]:
    if setting == "baseline":
        prompt = self.prompt_builder.build_baseline_prompt(row.query)
        return prompt, []

    if setting == "rag":
        chunk_indices = row.retrieved_chunks[:self.top_k]
        chunks = [row.candidate_chunks[i] for i in chunk_indices]
        prompt = self.prompt_builder.build_zero_shot_prompt(row.query, chunks)
        return prompt, chunk_indices

    if setting == "oracle":
        chunk_indices = self.retrieval_store.get_oracle_indices(row)
        chunks = [row.candidate_chunks[i] for i in chunk_indices]
        prompt = self.prompt_builder.build_zero_shot_prompt(row.query, chunks)
        return prompt, chunk_indices

  def retrieve_answers(
      self,
      df: pd.DataFrame,
      model,
      setting: str,
      thinking: bool = False,
    ):

    df = df.to_pandas() if not isinstance(df, pd.DataFrame) else df

    results = []

    for row in tqdm(df.itertuples(), total=len(df), desc=f"{setting} generation"):
      prompt, chunk_indices = self.build_prompt_from_row(row, setting)
      answer, _ = model.generate_answer(prompt, thinking)

      row_result = {
          "query_id": row.query_id,
          "retrieved_chunks": chunk_indices,
          "augmented_prompt": prompt,
          "generated_answer": answer,
      }

      # For evaluation
      row_result["short_answer"] = row.short_answer
      row_result["answer_pos"] = row.answer_pos
      row_result["query"] = row.query

      results.append(row_result)

    return results

## Utils

In [11]:
def export_jsonl(results: List[Dict], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True) if os.path.dirname(path) else None

    allowed_fields = [
        "query_id",
        "retrieved_chunks",
        "augmented_prompt",
        "generated_answer",
    ]

    with open(path, "w", encoding="utf-8") as f:
        for result in results:
            item = {
                field: result[field]
                for field in allowed_fields
                if field in result
            }
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


def make_submission_path(
    output_dir: str,
    split: str,
    variant: str,
) -> str:
    filename = f"doubleN-{split}-{variant}.jsonl"
    return os.path.join(output_dir, filename)

In [12]:
def display_results_table(results: List[Dict], n: int = 5):
    preview_df = pd.DataFrame(results[:n])

    cols = [
        "query_id",
        "query",
        "retrieved_chunks",
        "generated_answer",
        "short_answer",
    ]

    cols = [c for c in cols if c in preview_df.columns]

    display(
        preview_df[cols].style.set_properties(
            **{
                "white-space": "pre-wrap",
                "text-align": "left",
                "vertical-align": "top",
            }
        )
    )

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Experiments

In [ ]:
output_dir = "1_year/mnlp/hw_2"

In [14]:
data_module = DataModule(dev_size=0.2)
ds = data_module.load("sapienzanlp-course-materials/hw-mnlp-2026")

train_df, val_df = data_module.build_train_val_split()
test_df = data_module.to_dataframe("test")
blind_df = data_module.to_dataframe("blind")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

data/blind-00000-of-00001.parquet:   0%|          | 0.00/10.9M [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/66.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating blind split:   0%|          | 0/1322 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

In [51]:
retrieval_store = RetrievalStore()
prompt_builder = PromptBuilder()

In [16]:
test_ranking_path = f"/content/drive/MyDrive/{output_dir}/baseline1_test.jsonl"
blind_ranking_path = f"/content/drive/MyDrive/{output_dir}/baseline1_blind.jsonl"

baseline_test_df = retrieval_store.attach_rankings_from_jsonl(
    test_df,
    test_ranking_path,
)

baseline_blind_df = retrieval_store.attach_rankings_from_jsonl(
    blind_df,
    blind_ranking_path,
)

## [B1]: Implement at least two small LMs (<= 3B parameters) zero-shot in (a) baseline, (b) RAG and (c) Oracle settings

### Qwen3-0.6B

In [42]:
qwen3_06B_model_name = "Qwen/Qwen3-0.6B"

qwen3_06B_model = QALLM(qwen3_06B_model_name)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Baseline setting

In [53]:
runner = ExperimentRunner(
    prompt_builder,
    retrieval_store,
)

In [54]:
debug_results = runner.retrieve_answers(
    df=test_df.head(25),
    model=qwen3_06B_model,
    setting="baseline",
    thinking=False,
)

display_results_table(debug_results, n=25)

baseline generation:   0%|          | 0/25 [00:00<?, ?it/s]

,query_id,query,retrieved_chunks,generated_answer,short_answer
0,4HZKjht3X13n,where is the lady with an ermine located,[],The lady with an ermine is located in a fictional or imaginative setting.,['the National Museum in Kraków']
1,BeCayzNq9ZgD,lok sabha or rajya sabha which is bigger,[],rajya sabha,['Lok Sabha']
2,ZP7FeO0UVt3U,who was the actress who played pinky tuscadero on happy days,[],The actress who played Pinky Tuscadero on Happy Days is **Mia Farrow**.,['Roz Kelly']
3,Tt88r7IhbHtg,who wrote the song i want you back,[],I don't know.,['Berry GordyFreddie PerrenAlphonso MizellDeke Richards']
4,PblIzyQ705Kg,who wrote the music for fiddler on the roof,[],The music for *Fiddler on the Roof* was written by **David Hyman**.,['Jerry Bock']
5,EziXNbpoyZcf,who sang last night a dj saved my life,[],"The song ""Last Night a DJ Saved My Life"" was performed by **The Weeknd**.","['Réjane ""Reggie"" MagloireRose Marie Ramsey']"
6,9SjJ458p4ngc,where is the women's world cup being held,[],The Women's World Cup is held in Qatar.,['England']
7,UA3ho4s8iKcE,what city does big bang theory take place,[],Big Bang Theory takes place in New York.,"['Pasadena, California']"
8,PNtD2eXuCQev,when did the first goosebumps book come out,[],The first Goosebumps book came out in 2007.,['July 1992']
9,ONyEAVePMyZU,ethics comes from the greek word 'ethos' which means,[],ethos,"['""character""']"


In [22]:
results = runner.retrieve_answers(
    test_df,
    qwen3_06B_model,
    setting="baseline",
)

display_results_table(results, n=50)

KeyboardInterrupt: 

In [ ]:
export_jsonl(
    results,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-test-Qwen3-0.6B-baseline1-baseline.jsonl",
)

RAG setting

In [ ]:
qwen3_06B_rag_base_test = runner.retrieve_answers(
    df=baseline_test_df,
    model=qwen3_06B_model,
    setting="rag",
)

export_jsonl(
    qwen3_06B_rag_base_test,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-test-Qwen3-0.6B-baseline1-rag.jsonl",
)

In [ ]:
qwen3_06B_rag_base_blind = runner.retrieve_answers(
    df=baseline_blind_df,
    model=qwen3_06B_model,
    setting="rag",
)

export_jsonl(
    qwen3_06B_rag_base_blind,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-blind-Qwen3-0.6B-baseline1-rag.jsonl",
)

Oracle setting

In [ ]:
qwen3_06B_orac_base_test = runner.retrieve_answers(
    df=baseline_test_df,
    model=qwen3_06B_model,
    setting="oracle",
)

display_results_table(qwen3_06B_orac_base_test, n=5)

### SmolLM2-1.7B

In [ ]:
smollm2_1_7B_model_name = "HuggingFaceTB/SmolLM2-1.7B"

smollm2_1_7B_model = SmolLM(smollm2_1_7B_model_name)

Baseline setting

In [ ]:
smollm2_1_7B_base_test = runner.retrieve_answers(
    test_df,
    smollm2_1_7B_model,
    setting="baseline",
)

display_results_table(smollm2_1_7B_base_test, n=5)

RAG setting

In [ ]:
smollm2_1_7B_rag_base_test = runner.retrieve_answers(
    df=test_df,
    model=smollm2_1_7B_model,
    setting="rag",
)

export_jsonl(
    qwen3_06B_rag_base_test,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-test-SmolLM2-1.7B-baseline1-rag.jsonl",
)

In [ ]:
smollm2_1_7B_rag_base_blind = runner.retrieve_answers(
    df=test_df,
    model=smollm2_1_7B_model,
    setting="rag",
)

export_jsonl(
    qwen3_06B_rag_base_blind,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-blind-SmolLM2-1.7B-baseline1-rag.jsonl",
)

Oracle setting

In [ ]:
smollm2_1_7B_orac_base_test = runner.retrieve_answers(
    df=test_df,
    model=smollm2_1_7B_model,
    setting="oracle",
)

display_results_table(qwen3_06B_orac_base_test, n=5)

## [B2]: Evaluation Framework